# Khipus.ai — Estadística Aplicada con Python

## Módulo 2 · Clase 1 — Estadística Descriptiva
### Dataset: `airnb.csv` (538 anuncios de Airbnb)

**Docente:** Walter J. Méndez · UTEPSA / Khipus.ai

**Nombre:** _______________________________________________

---

### 🧭 La pregunta que vamos a responder hoy

> ## ¿Cuánto cuesta *realmente* un Airbnb?

Parece una pregunta boba. No lo es. Vamos a necesitar **toda** la estadística descriptiva para
responderla bien, y en el camino vamos a descubrir que la respuesta más obvia (*"saco el promedio y listo"*)
es justamente la equivocada.

---

### ¿Qué vas a saber hacer al final de esta clase?

| Concepto estadístico | Herramienta en Python |
|---|---|
| Tablas de frecuencia | `value_counts()` |
| Media, mediana, moda | `.mean()` `.median()` `.mode()` |
| Rango, varianza, desviación estándar | `.max()-.min()` `.var()` `.std()` |
| Percentiles y cuartiles | `.quantile()` |
| Resumen completo | `.describe()` |
| Distribución y forma | histograma, boxplot, `.skew()` |
| Comparación entre grupos | `.groupby()` |
| Relación entre variables | scatter plot, `.corr()` |

Y algo que no está en ningún temario oficial pero que es lo más importante:
**darte cuenta cuándo un número te está mintiendo.**

---

### 📌 Cómo usar este notebook

- 👨‍🏫 **Explicación docente**: lo vemos juntos en clase. Las celdas ya vienen resueltas.
- ✍️ **Celdas `TU TURNO`**: las completás vos. Tienen pistas.
- ⭐ **Desafío opcional**: para quien quiera ir más lejos. No es obligatorio.
- 🤔 **Celdas de predicción**: **antes de ejecutar**, escribí tu respuesta en el chat de Teams.
  No es un juego: predecir antes de ver el resultado es lo que hace que el concepto se te quede.


---

# 0. La historia detrás de `airnb.csv`

Antes de tocar una sola línea de código, conviene saber **de dónde salió** el archivo con el que
vamos a trabajar. En el mundo real esta pregunta se hace siempre. Y cuando no se hace, pasan desgracias.

### El origen

Este dataset vive en Kaggle, publicado por un usuario llamado **joyshil0599**. En la plataforma
aparece como *"Airbnb Listing Data 2023"* (la dirección web usa otro nombre,
*airbnb-listing-data-for-data-science*, lo cual ya es un primer aviso de que en datos reales
nada está prolijo). Son **538 anuncios**, de todo el mundo: Estados Unidos, Indonesia, Tailandia,
Canadá, Filipinas, Reino Unido, Italia, México y Grecia. Bolivia no, lamentablemente.

### Primera señal de alerta 🚩

El archivo se llama **`airnb.csv`**.

No `airbnb.csv`. **`airnb.csv`**. Le falta una `b`.

Así nos llega el archivo desde el material del curso, con el error de tipeo incluido, y así lo
vamos a usar nosotros. **Bienvenidos a los datos reales.** En los cursos los datasets vienen limpios
y con nombres bonitos. En el trabajo te llegan archivos llamados
`reporte_final_v2_CORREGIDO_ahora_si (3).xlsx`.

### Lo que este dataset NO tiene

- ❌ **No tiene diccionario de datos.** Nadie documentó qué significa cada columna.
- ❌ **No sabemos exactamente cuándo se scrapeó.** El título dice 2023, pero la columna `Date`
  del archivo **ni siquiera trae el año** (`"Jun 11 - 16"`), así que no lo podemos confirmar desde los datos.
- ❌ **No sabemos cómo se eligieron esos 538 anuncios.** ¿Son aleatorios? ¿Son los primeros que
  aparecieron? ¿Son los más baratos? **No lo sabemos.**

Ese último punto es serio y lo vamos a repetir al final de la clase: **todo lo que calculemos hoy
describe a estos 538 anuncios**, no a "los Airbnb del mundo". Describir ≠ generalizar.

### Un dato curioso para el final de la clase

En Kaggle hay notebooks públicos que hacen exactamente lo que nosotros vamos a hacer en la Sección 3:
separar las columnas sucias en columnas limpias. Un enfoque muy común es buscar el **tipo de cama**
(*queen*, *king*, *sofa*) dentro del texto, y armar una columna `Country` partiendo el título en
tres partes separadas por comas.

Con **este** archivo, ese enfoque falla — y lo vamos a verificar nosotros mismos más adelante:

- Filas donde la columna `Number of bed` menciona un tipo de cama: **cero**. Todas dicen simplemente
  `"2 beds"`, `"3 beds"`. (Sí aparecen *queen*/*king*/*sofa* en la columna `Detail`, en 31 filas —
  pero esa no es la columna de la que ese método intenta sacar el tipo de cama.)
- Títulos que **no** tienen las tres partes que ese método necesita: **483 de 538, el 90%**.

Y lo más instructivo: **ese código no se caería**. No lanzaría ningún error. Simplemente produciría
una columna llena de valores nulos, en silencio.

No es que el autor sea malo. Es que **los datos cambian y el código queda viejo**. Guardate esa idea:
es la razón por la que existe el control de versiones que vieron en el Módulo 1.

> 📌 **Nota de honestidad intelectual, que también es contenido de la clase:** los dos números de
> arriba (cero tipos de cama, 483 títulos) están **verificados sobre el archivo que tenemos en la mano**
> — los vas a poder reproducir vos. Lo que *no* podemos verificar es el código exacto de ningún
> notebook ajeno. Por eso hablamos de "un enfoque común" y no le adjudicamos a nadie en particular
> algo que no leímos. **Afirmá lo que verificaste; para lo demás, bajá el tono.**

---


---

# 0.4 · ¿Qué es la estadística?

Antes del mapa del módulo, la pregunta que está más arriba de todas. Vas a usar la palabra
"estadística" durante las próximas semanas: conviene saber qué nombra.

> ## La estadística es la ciencia de extraer conocimiento a partir de datos que varían.

La palabra que carga todo el peso es **varían**. Y es la que casi nunca se enseña bien.

## Por qué existe

Si los 538 anuncios de este archivo costaran todos lo mismo, la estadística no existiría: mirás uno,
sabés todo, terminaste.

Pero cuestan **entre $16 y $955**. Y esa variación crea un problema que ninguna otra disciplina
resuelve:

> **¿Cómo decís algo verdadero sobre un conjunto cuyos elementos son todos distintos entre sí?**

La estadística es el conjunto de métodos que inventamos para responder eso. Usa matemática como
herramienta —igual que la física—, pero su objeto de estudio no son los números: **es la variación y
la incertidumbre**.

## Los dos movimientos

Toda la disciplina se reduce a dos:

| Movimiento | Su pregunta | Cuánta certeza da |
| --- | --- | --- |
| **Describir** | ¿Qué hay en estos datos? | **Total.** Es aritmética, no opinión. |
| **Inferir** | ¿Qué puedo afirmar sobre lo que **no** medí? | **Parcial — pero medida.** |

Y acá está lo más elegante de la disciplina: en el segundo movimiento, su mayor contribución no es
darte la respuesta, sino **decirte cuánta confianza merece esa respuesta**.

> Otras disciplinas te dan una respuesta.
> La estadística te da una respuesta **y el tamaño de su margen de error**.

*(El mapa de acá abajo muestra **tres** etapas, no dos: el modelado de la Clase 3 es la extensión del
segundo movimiento — infiere sobre casos nuevos en vez de sobre una población.)*

## Lo que la estadística NO es

**No es "trabajar con números".** Un contador trabaja con números todo el día y no hace estadística:
sus cifras son exactas y no varían.

**No es acumular datos.** Tener 538 anuncios no es hacer estadística. Preguntarles algo, sí.

**No es un conjunto de fórmulas.** Las fórmulas son la parte automatizable, la que pandas ejecuta en
un milisegundo. Lo que **no** se automatiza es decidir *qué* medir, *cuál* medida usar y *qué
significa* el resultado.

> Esta clase entera es un caso de eso: `.mean()` y `.median()` son **igual de fáciles de escribir**.
> Elegir cuál de las dos reportar es el 90% del trabajo intelectual.

## Para llevarse

> **La estadística es lo que hacemos cuando las cosas no son todas iguales.** Si todos los clientes
> compraran lo mismo, si todos los pacientes reaccionaran igual, si todos los alumnos rindieran
> igual, no haría falta. Existe porque el mundo varía — y su trabajo es **encontrar el patrón sin
> negar la excepción**.

Guardá esa última frase, porque hoy vas a hacer las dos cosas. El patrón lo vamos a encontrar. Y las
excepciones también: en este archivo hay anuncios de **$955**. No son errores — son casas de lujo
reales. La estadística no las borra: las reporta aparte.

# 0.5 · El mapa: ¿dónde está parada esta clase?

Antes de calcular nada, ubicate. Este módulo tiene **tres clases** y hoy es la primera. Vale la pena
saber qué hace cada una, porque **la estadística descriptiva se entiende mucho mejor cuando sabés qué
es lo que NO hace**.

## Las tres etapas del módulo

```
                        ESTADÍSTICA
                             │
        ┌────────────────────┼────────────────────┐
        │                    │                    │
   DESCRIPTIVA          INFERENCIAL          MODELADO
   (Clase 1 · HOY)      (Clase 2 · 01/08)   (Clase 3 · 08/08)
        │                    │                    │
   Resume lo que      Generaliza de la      Predice y explica
    ya tenemos       muestra a la población   con un modelo
        │                    │                    │
   "Esto es lo         "Esto es lo que      "Esto es lo que
    que hay"           probablemente sea     va a pasar si
                       cierto en general"    cambian las condiciones"
```

## En qué se diferencian, concretamente

| Criterio | **Descriptiva** (hoy) | **Inferencial** (Clase 2) | **Modelado / Regresión** (Clase 3) |
|---|---|---|---|
| **Su pregunta** | ¿Qué hay en estos datos? | ¿Qué puedo afirmar sobre lo que NO medí? | ¿Cuánto valdrá Y para un caso nuevo? |
| **Dirección** | Datos → resumen | Muestra → población | Datos → predicción |
| **Alcance** | Solo la muestra | La población entera | Casos nuevos |
| **Incertidumbre** | No aplica | **Se cuantifica** | Se cuantifica |
| **Supuestos** | **Casi ninguno** | Muestreo aleatorio, a veces normalidad | Varios y fuertes (linealidad, etc.) |
| **Herramientas** | media, mediana, desviación, cuartiles, histograma, boxplot | Teorema Central del Límite, intervalos de confianza, pruebas t/z, valor-p | regresión lineal, regresión logística |
| **Puede fallar por** | Mala interpretación | Muestra sesgada | Supuestos violados |

### El mismo ejemplo, en las tres etapas

- **Descriptiva (hoy):** *"La mediana de precio de estos 538 anuncios es $138, con un rango típico de $90 a $222."*
  → **Certeza total, dentro de la muestra.** El $138 es un hecho aritmético, no una estimación.

- **Inferencial (Clase 2):** *"Con estos 538 anuncios como muestra, estimamos que el precio mediano de
  todos los Airbnb del mundo está entre $X e $Y, con 95% de confianza."*
  → **Certeza parcial, pero cuantificada.** Esa es su virtud: no elimina la incertidumbre, la mide.

- **Modelado (Clase 3):** *"¿Puedo predecir el precio de un anuncio a partir del número de camas, el
  tipo de propiedad y la ubicación?"*
  → **Un modelo que se puede aplicar a un anuncio que todavía no existe.**

## 💡 La ventaja subestimada de la descriptiva

Mirá la fila de **supuestos** en la tabla.

La estadística descriptiva es la única de las tres que **casi no asume nada**. La media de nuestros 538
precios es $175.26 y punto: es un hecho aritmético, no una estimación que pueda estar equivocada.

Las otras dos asumen cosas —que la muestra es aleatoria, que la distribución es normal, que la relación
es lineal— y **cuando el supuesto falla, el resultado es basura con aspecto profesional.**

> ## 💡 La idea que quiero que te lleves de esta sección
> ### La descriptiva es la etapa más humilde y la más confiable.
> ### Y es la que **valida los supuestos de todas las demás.**

Concretamente, todo lo que hagamos hoy vuelve a aparecer:

| La clase que viene... | ...va a necesitar esto de hoy |
|---|---|
| Inferencial (01/08) | **La media y la desviación estándar son los insumos de casi toda prueba** |
| Inferencial (01/08) | Verificar si los datos son normales **antes** de elegir el método (lo hacemos en la Sección 6) |
| Regresión (08/08) | **La correlación es el paso previo.** Sin correlación, no hay modelo lineal útil (Sección 9) |
| Regresión (08/08) | Los outliers distorsionan una regresión igual que distorsionan una media (Sección 6) |

**La estadística descriptiva no es una introducción que después se abandona: es la base sobre la que se
apoyan las otras dos.**

---


# 1. Primer contacto con los datos

## 👨‍🏫 Regla de oro número uno

> **Nunca calcules un promedio sobre datos que no miraste.**

Es la regla más violada de la ciencia de datos. Alguien recibe un CSV, escribe `.mean()`, pega el
número en un PowerPoint, y a la semana está explicando en una reunión por qué su cifra decía que
el cliente promedio tiene -3 años de edad.

Antes de calcular: **cargar, mirar, y entender la forma del archivo.**

---

## 👨‍🏫 Las tres librerías de hoy

Hoy usamos tres, y conviene entender **qué hace cada una y por qué la necesitamos**, en vez de
copiar los `import` de memoria.

### 🐼 pandas — la que hace el 85% del trabajo

**Qué es:** la librería para manipular y analizar **datos tabulares** (filas y columnas). La creó
Wes McKinney en 2008. El nombre viene de *panel data*, un término de econometría, no del animal.

**Su estructura central es el `DataFrame`:** una tabla con filas y columnas etiquetadas.
Conceptualmente, *una hoja de Excel manejada con código en vez de con el mouse*.

**Cómo se invoca:**
```python
import pandas as pd     # el alias `pd` es una convención universal: usalo siempre
```

**Qué funciones suyas vamos a tocar hoy:**

| Para qué | Función |
|---|---|
| Cargar el CSV | `pd.read_csv()` |
| Ver la estructura | `.shape` · `.head()` · `.info()` · `.columns` |
| Diagnosticar calidad | `.isnull().sum()` · `.duplicated()` · `.drop_duplicates()` |
| Extraer datos del texto | `.str.split()` · `.str.extract()` |
| Convertir tipos con tolerancia | `pd.to_numeric(errors="coerce")` |
| Tendencia central | `.mean()` · `.median()` · `.mode()` |
| Dispersión | `.var()` · `.std()` · `.quantile()` |
| Forma | `.skew()` |
| Resumen integral | `.describe()` |
| Frecuencias | `.value_counts()` |
| Comparar grupos | `.groupby().agg()` |
| Relación entre variables | `.corr()` |

### 🔢 numpy — el motor de abajo

**Qué es:** la librería base de cálculo numérico de Python. Aporta el `ndarray`, un arreglo
eficiente implementado en C.

**Su rol hoy es casi invisible, y por eso mismo importante:** cada vez que pandas calcula una media
o una desviación, **por debajo está numpy**. Vos no lo llamás; pandas lo llama por vos. Es la base
de todo el ecosistema científico de Python (pandas, matplotlib, scikit-learn, PyTorch).

**Cómo se invoca:**
```python
import numpy as np
```

**Dónde SÍ lo usamos explícitamente hoy:** `np.sqrt()` (verificar que √varianza = desviación),
y `np.linspace()` / `np.exp()` / `np.pi` (dibujar la curva normal teórica).

### 📊 matplotlib — la visualización

**Qué es:** la librería de gráficos fundamental de Python. La creó John Hunter en 2003, un
neurobiólogo que quería replicar en Python las capacidades gráficas de MATLAB.

**Cómo se invoca:**
```python
import matplotlib.pyplot as plt    # `pyplot` es el submódulo de trazado; el alias `plt` es la convención
%matplotlib inline                 # magia de Jupyter: dibuja los gráficos DENTRO del notebook
```

**Qué funciones suyas vamos a tocar hoy:**

| Gráfico | Función |
|---|---|
| Histograma | `plt.hist()` |
| Boxplot | `plt.boxplot()` |
| Dispersión | `plt.scatter()` |
| Línea (curva normal) | `plt.plot()` |
| Barras | `.plot(kind="bar")` (vía pandas) |
| Torta | `.plot(kind="pie")` (vía pandas) |
| Línea vertical de referencia | `plt.axvline()` |
| Rótulos y leyenda | `plt.title()` · `plt.xlabel()` · `plt.ylabel()` · `plt.legend()` |

> 💡 **¿Y seaborn?** Existe, y hace gráficos más lindos con menos código. **No la usamos hoy a
> propósito:** todo el material oficial del repositorio de Khipus usa matplotlib, y queremos que lo
> que aprendas acá sea idéntico a lo que veas allá. Más adelante en el módulo la vas a conocer.

> ⚠️ **matplotlib es potente y verboso.** Hacer un gráfico simple lleva varias líneas. Lo vas a sufrir
> en la Sección 8 con el boxplot comparativo — y ahí vas a entender por qué existe seaborn.


In [1]:
# Las tres librerías de la clase
import pandas as pd            # manejo de tablas
import numpy as np             # operaciones numéricas
import matplotlib.pyplot as plt  # gráficos

# Para que los gráficos aparezcan dentro del notebook
%matplotlib inline

print("✅ Librerías cargadas. Empezamos.")

✅ Librerías cargadas. Empezamos.


## Cargar el archivo

`pd.read_csv()` lee un archivo separado por comas y devuelve un **DataFrame**: la estructura de datos
más importante de pandas. Pensalo como una hoja de Excel, pero manejada con código en vez de con el mouse.

La celda de abajo busca el archivo **en la carpeta `datos/` del repositorio** y, si no lo encuentra
(por ejemplo, si estás corriendo esto en **Google Colab**), lo descarga directo del repositorio
público de Khipus. Así el notebook corre en cualquier lado sin que tengas que tocar nada.


In [2]:
# Celda de carga estándar: usa el archivo local si existe; si no, lo baja del repo oficial.
# Así el notebook corre igual en tu máquina y en Google Colab.
import os

_LOCAL = "../datos/airnb.csv"
_URL = "https://raw.githubusercontent.com/Khipus-ai/Applied_Statistics_Python/main/1%20Descriptive_Statistics/airnb.csv"

_origen = _LOCAL if os.path.exists(_LOCAL) else _URL
df = pd.read_csv(_origen)

print(f"✅ Archivo cargado desde: {'la carpeta local datos/' if _origen == _LOCAL else 'el repositorio de Khipus (GitHub)'}")


✅ Archivo cargado desde: el repositorio de Khipus (GitHub)


## ¿Qué tamaño tiene?

`.shape` devuelve `(filas, columnas)`. Es lo primero que se mira siempre.

In [3]:
df.shape

(538, 7)

**538 filas y 7 columnas.**

Es un dataset **chico**. Y eso está bien: cabe en la cabeza, todo se ejecuta instantáneamente,
y podés revisar los resultados a mano si dudás.

⚠️ **Ojo con una confusión clásica:** esto **no es Big Data**. En el Módulo 1 vimos que Big Data
implica volumen, velocidad y variedad. Acá tenemos 56 kilobytes. Eso entra en un correo electrónico.
Estadística descriptiva se hace sobre datos de cualquier tamaño.

## Mirar las primeras filas

`.head()` muestra las primeras 5 filas. Es la forma más rápida de entender *qué contiene* cada columna.

In [4]:
df.head()

,Title,Detail,Date,Price(in dollar),Offer price(in dollar),Review and rating,Number of bed
0,"Chalet in Skykomish, Washington, US",Sky Haus - A-Frame Cabin,Jun 11 - 16,306,229.0,4.85 (531),4 beds
1,"Cabin in Hancock, New York, US",The Catskill A-Frame - Mid-Century Modern Cabin,Jun 6 - 11,485,170.0,4.77 (146),4 beds
2,"Cabin in West Farmington, Ohio, US",The Triangle: A-Frame Cabin for your city retreat,Jul 9 - 14,119,522.0,4.91 (515),4 beds
3,"Home in Blue Ridge, Georgia, US",*Summer Sizzle* 5 Min to Blue Ridge* Pets* Hot...,Jun 11 - 16,192,348.0,4.94 (88),5 beds
4,"Chalet in Grand Étang, Canada",• Cedar Peak • 2 Bedroom Barrier-Free Chalet,Jun 8 - 15,381,NaN,5.0 (48),2 beds


## 🤔 Momento de observación

Mirá bien esa tabla antes de seguir. Sin ejecutar nada más, **¿qué problemas ves?**

Tomate 30 segundos. Escribí en el chat lo que te llame la atención.

<details>
<summary>👉 Click acá recién DESPUÉS de haber pensado</summary>

- `Title` mezcla **tipo de propiedad + ciudad + estado + país** en una sola celda.
- `Date` es un rango de fechas escrito como texto: `"Jun 11 - 16"`. Ni siquiera dice el año.
- `Review and rating` mete **dos variables distintas** en una celda: `"4.85 (531)"` es la calificación
  Y la cantidad de reseñas.
- `Number of bed` dice `"4 beds"` — es texto, no un número.
- `Offer price(in dollar)` tiene un montón de `NaN` (valores vacíos).
- Los nombres de las columnas tienen **paréntesis y espacios**, lo cual es incomodísimo para escribir.

Si viste al menos tres, tenés buen ojo. Si no viste ninguno, tranquilo: para eso es la clase.
</details>

## La radiografía: `.info()`

`.info()` es la herramienta diagnóstica más útil de pandas. En una sola salida te dice:
cuántas filas hay, qué **tipo de dato** tiene cada columna, y **cuántos valores no nulos** tiene cada una.

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 538 entries, 0 to 537
Data columns (total 7 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Title                   538 non-null    str    
 1   Detail                  538 non-null    str    
 2   Date                    538 non-null    str    
 3   Price(in dollar)        538 non-null    int64  
 4   Offer price(in dollar)  95 non-null     float64
 5   Review and rating       536 non-null    str    
 6   Number of bed           538 non-null    str    
dtypes: float64(1), int64(1), str(5)
memory usage: 29.6 KB


## 👨‍🏫 Cómo leer esta salida

Dos columnas de esa tabla merecen atención:

**`Dtype` (tipo de dato):**

| Dtype | Qué es | ¿Se puede promediar? |
|---|---|---|
| `int64` | Número entero | ✅ Sí |
| `float64` | Número decimal | ✅ Sí |
| `str` | Texto | ❌ No |

Mirá: de 7 columnas, **solo 2 son numéricas**. Las otras 5 son texto. Si intentás sacar el promedio
de `Number of bed`, Python te va a mirar con cara rara.

> ⚠️ **Detalle de versiones que te va a ahorrar una confusión.** Nosotros estamos usando **pandas 3.x**,
> que muestra las columnas de texto como **`str`**. Si abrís un tutorial, un video o el notebook oficial
> de Khipus —hechos con **pandas 2.x o anterior**— vas a ver **`object`** en ese mismo lugar.
>
> **Son lo mismo:** ambos significan "esta columna es texto". `object` era el nombre viejo y genérico;
> `str` es el tipo dedicado que pandas 3.0 activó por defecto. No te asustes si los ves mezclados.

**`Non-Null Count`:**

`Offer price(in dollar)` tiene apenas **95 valores no nulos** de 538. Eso ya es una alarma que vamos
a investigar en la próxima sección.

### 🎁 La buena noticia del día

`Price(in dollar)` es **`int64`**. Está limpio. No hay que convertir nada, no hay símbolos de dólar
pegados, no hay comas de miles. Nuestra variable protagonista viene lista para usar.

Esto casi nunca pasa. Disfrutalo.


## ✍️ TU TURNO 1

Mostrá los **nombres de todas las columnas** del DataFrame.

💡 *Pista: hay un atributo del DataFrame que se llama exactamente como lo que estás buscando (en inglés).*

In [ ]:
# TU TURNO 1: mostrar los nombres de las columnas
# Tu código aquí



---

# 2. Diagnóstico de calidad: ¿podemos confiar en este archivo?

## 👨‍🏫 Regla de oro número dos

> **Antes de analizar los datos, hay que auditarlos.**

Un dataset puede estar perfectamente formateado, cargar sin errores, tener todos los tipos correctos...
y aun así estar lleno de basura. Pandas no te va a avisar. Pandas ejecuta lo que le pidas, aunque no
tenga ningún sentido.

Vamos a hacerle tres preguntas al archivo.

## Pregunta 1: ¿cuántos datos faltan?

In [ ]:
# .isnull() marca True donde hay un valor vacío
# .sum() cuenta cuántos True hay en cada columna
df.isnull().sum()

## 👨‍🏫 El caso de `Offer price`

**443 valores faltantes de 538.** Sacá la cuenta: eso es el **82%** de la columna vacía.

Vamos a calcular el porcentaje, porque un número absoluto sin contexto no dice nada:

In [ ]:
# Porcentaje de valores faltantes por columna, ordenado de peor a mejor
porcentaje_nulos = (df.isnull().sum() / len(df) * 100).round(1)
porcentaje_nulos.sort_values(ascending=False)

### ⚖️ La decisión

**`Offer price` tiene 82.3% de datos faltantes.**

¿Qué se hace con una columna así? En la práctica profesional se maneja un criterio como este:

| % de nulos | Qué se suele hacer |
|---|---|
| < 5% | Se rellenan o se eliminan esas filas |
| 5% – 30% | Se analiza caso por caso, con cuidado |
| > 50% | **La columna se descarta**, salvo que sea crítica |

> ⚠️ **Importante, y esto anotalo:** esta tabla es una **regla de oficio, no una norma**. No la busques
> en un libro ni la cites como si fuera un estándar, porque no existe una fuente autoritativa que fije
> esos cortes. Cada industria los mueve: en un ensayo clínico un 5% de faltantes ya es grave; en datos
> de sensores industriales un 30% puede ser rutina. **Es criterio acumulado, y sirve para orientarte,
> no para decidir por vos.**

Con 82% vacío, cualquier conclusión que saquemos de `Offer price` estaría basada en menos de 1 de cada
5 anuncios. **Y no sabemos si esos 95 son representativos.**

Quizá solo los anuncios caros tienen descuento. Quizá el scraper falló en ciertas páginas. Quizá
Airbnb solo muestra ofertas en algunas fechas. No lo sabemos, y no hay diccionario de datos que nos lo diga.

> 💡 **Regla:** un dato faltante nunca es aleatorio hasta que se demuestre lo contrario.

## Pregunta 2: ¿hay filas repetidas?


In [ ]:
# .duplicated() marca True las filas que son copia exacta de otra anterior
df.duplicated().sum()

### 😐 24 filas duplicadas

Veinticuatro anuncios están cargados **dos veces**, idénticos en las 7 columnas.

¿Por qué importa? Porque **cada duplicado vota dos veces** en todos nuestros cálculos. Si un anuncio
carísimo está duplicado, empuja el promedio hacia arriba con el doble de fuerza que los demás.

Veamos algunos: 

In [ ]:
# keep=False marca TODAS las apariciones, no solo las repeticiones
duplicados = df[df.duplicated(keep=False)].sort_values("Title")
print(f"Filas involucradas en duplicación: {len(duplicados)}")
duplicados.head(6)

### ¿Cuánto cambia el resultado si los eliminamos?

Esta es la pregunta correcta. No hay que asustarse por un problema antes de medir su impacto.

In [ ]:
df_sin_dup = df.drop_duplicates()

print(f"Filas originales      : {len(df)}")
print(f"Filas sin duplicados  : {len(df_sin_dup)}")
print()
print(f"Precio medio CON duplicados : ${df['Price(in dollar)'].mean():.2f}")
print(f"Precio medio SIN duplicados : ${df_sin_dup['Price(in dollar)'].mean():.2f}")
print()
print(f"Mediana CON duplicados : ${df['Price(in dollar)'].median():.2f}")
print(f"Mediana SIN duplicados : ${df_sin_dup['Price(in dollar)'].median():.2f}")

### 👨‍🏫 Interpretación

La diferencia es de menos de un dólar. **En este caso los duplicados casi no afectan el resultado.**

Y esa es una lección tan valiosa como la anterior: **detectar un problema y medir que es irrelevante
también es hacer bien el trabajo.** No todo hallazgo obliga a actuar.

Pero fijate que **solo lo sabemos porque lo medimos**. Si no revisábamos, hoy estaríamos reportando
un número sin saber si era confiable.

> ⚠️ **Nota metodológica:** para el resto de la clase vamos a seguir usando `df` (con duplicados),
> porque queremos que nuestros números coincidan con el notebook oficial del repositorio de Khipus.
> En un trabajo real, los eliminarías.

## Pregunta 3: ¿los datos son lógicamente coherentes?

Esta es la auditoría que casi nadie hace, y es la que más problemas encuentra.

`Offer price` debería ser un **precio de oferta**. Una oferta, por definición, es **más barata** que
el precio original. Verifiquémoslo.

In [ ]:
# Nos quedamos solo con las filas que SÍ tienen precio de oferta
con_oferta = df.dropna(subset=["Offer price(in dollar)"])

# ¿En cuántas la "oferta" es MÁS CARA que el precio original?
oferta_mas_cara = con_oferta["Offer price(in dollar)"] > con_oferta["Price(in dollar)"]

print(f"Anuncios con precio de oferta      : {len(con_oferta)}")
print(f"Ofertas MÁS CARAS que el precio 😱 : {oferta_mas_cara.sum()}")
print(f"Porcentaje absurdo                 : {oferta_mas_cara.sum() / len(con_oferta) * 100:.1f}%")

## 🤯 La mitad de las "ofertas" son más caras

**48 de 95.** Más de la mitad de los descuentos de este dataset consisten en cobrarte más.

Es el equivalente estadístico de esos carteles de liquidación donde primero suben el precio y después
lo tachan. Descuento estilo Black Friday: *"antes 119, AHORA 522"*.

Veamos los casos más escandalosos: 

In [ ]:
caso_absurdo = con_oferta[oferta_mas_cara].copy()
caso_absurdo["Sobreprecio"] = (
    caso_absurdo["Offer price(in dollar)"] - caso_absurdo["Price(in dollar)"]
)

caso_absurdo.nlargest(5, "Sobreprecio")[
    ["Title", "Price(in dollar)", "Offer price(in dollar)", "Sobreprecio"]
]

## 👨‍🏫 La lección más importante de esta sección

Mirá lo que acaba de pasar:

- El dato **existe** ✅
- El dato es **numérico** ✅
- El dato **se puede promediar sin errores** ✅
- El dato **está mal** ❌

**Python no tiene forma de saberlo.** `pandas` va a calcular felizmente el promedio de esa columna y
te va a devolver un número con dos decimales que parece muy profesional.

> ## 💡 Frase para memorizar
> ### El único filtro contra un dato absurdo es una persona que conozca el negocio.

Esto conecta directo con el Módulo 1: **conocimiento del dominio**. Si no supieras qué es un descuento,
no habrías notado nada raro. El estadístico que no entiende el negocio produce números correctos e
inútiles.

## ✍️ TU TURNO 2

Contá **cuántos anuncios tienen exactamente el mismo precio de oferta que el precio original**
(descuento de cero, la oferta más honesta del dataset).

💡 *Pista: usá `con_oferta` y comparalos con `==` en vez de `>`. Después `.sum()`.*

In [ ]:
# TU TURNO 2: ofertas con descuento igual a cero
# Tu código aquí



---

# 3. Rescatando variables escondidas en el texto

## El problema

Tenemos 538 anuncios y **solo 2 columnas numéricas**. Con eso no se puede hacer mucha estadística.

Pero mirá bien: hay información numérica **escondida dentro de las columnas de texto**.

| Columna de texto | Información escondida |
|---|---|
| `"Chalet in Skykomish, Washington, US"` | El **tipo de propiedad**: Chalet |
| `"4 beds"` | El **número de camas**: 4 |
| `"4.85 (531)"` | La **calificación**: 4.85 · Las **reseñas**: 531 |

Sacar esa información se llama **extracción de variables** (*feature extraction*), y es uno de los
pasos que más valor agrega en un proyecto real: estás **creando** variables que antes no existían.

## Herramienta: el accesorio `.str`

Cuando una columna de pandas contiene texto, `.str` te da acceso a todas las operaciones de texto de
Python, **aplicadas a las 538 filas de una sola vez**. Sin bucles.

```python
df["columna"].str.split(" ")   # partir el texto
df["columna"].str[0]           # quedarse con un pedazo
```

> 💡 Esto se llama **vectorización**: pandas no recorre las filas una por una, delega la operación a
> numpy, que la ejecuta sobre todo el arreglo de golpe en código C compilado. Es la razón por la que
> es órdenes de magnitud más rápido que un `for`.

## Extracción 1: el número de camas

`"4 beds"` → partimos por el espacio → `["4", "beds"]` → nos quedamos con el primero → `"4"` →
lo convertimos a entero → `4`.


In [ ]:
# Paso a paso, para que se vea la mecánica

# Paso 1: partir el texto por el espacio
paso1 = df["Number of bed"].str.split(" ")
print("Paso 1 — el texto partido en pedazos:")
print(paso1.head(3).tolist())

# Paso 2: quedarnos con el primer pedazo
paso2 = paso1.str[0]
print("\nPaso 2 — solo el primer pedazo (todavía es TEXTO):")
print(paso2.head(3).tolist())

# Paso 3: convertirlo a número entero
paso3 = paso2.astype(int)
print("\nPaso 3 — convertido a NÚMERO:")
print(paso3.head(3).tolist())

In [ ]:
# Todo junto en una sola línea, y lo guardamos como columna nueva
df["camas"] = df["Number of bed"].str.split(" ").str[0].astype(int)

df[["Number of bed", "camas"]].head()

## Extracción 2: el tipo de propiedad

Los títulos siguen el patrón `"<tipo> in <lugar>"`:

- `"Chalet in Skykomish, Washington, US"`
- `"Cabin in Hancock, New York, US"`
- `"Room in Mexico City, Mexico"`

Partimos por `" in "` y nos quedamos con lo de la izquierda.

In [ ]:
df["tipo"] = df["Title"].str.split(" in ").str[0]

df[["Title", "tipo"]].head()

## Extracción 3: la calificación... y acá se rompe todo 💥

`"4.85 (531)"` → partimos por el espacio → nos quedamos con `"4.85"` → convertimos a decimal.

Es la misma receta que usamos con las camas. **¿Qué podría salir mal?**

🤔 **Predicción:** antes de ejecutar la celda de abajo, ¿creés que va a funcionar? Escribí SÍ o NO en el chat.

In [ ]:
# Intentamos la misma receta que funcionó con las camas
try:
    df["rating"] = df["Review and rating"].str.split(" ").str[0].astype(float)
    print("Funcionó sin problemas 🎉")
except ValueError as error:
    print("💥 ERROR — Python no pudo hacerlo:")
    print()
    print(f"   ValueError: {error}")
    print()
    print("👆 Leé el mensaje. Python te está diciendo EXACTAMENTE cuál es el problema.")

## 👨‍🏫 Cómo leer un error (la habilidad más subestimada)

```
ValueError: could not convert string to float: 'New'
```

Traducido: *"no pude convertir el texto `'New'` en un número decimal"*.

Python no está siendo críptico. Te dice:
1. **Qué tipo de error es** → `ValueError`, un valor inapropiado
2. **Qué intentaba hacer** → convertir texto a decimal
3. **Cuál valor exacto lo rompió** → `'New'`

> 💡 **La enorme mayoría de los errores de Python se resuelven leyendo el mensaje completo.**
> No lo cierres. No entres en pánico. Leelo.

Esta es probablemente la habilidad más transferible de toda la clase: el principiante ve un error y
se bloquea; quien tiene oficio lo lee de abajo hacia arriba y sigue trabajando.

## ¿Qué es ese `'New'`?

Son anuncios **nuevos, que todavía no tienen calificación**. En vez de dejar la celda vacía, quien
armó el dataset escribió la palabra `"New"`.

Buscámoslos:


In [ ]:
# Buscamos todos los valores que NO empiezan con un dígito
no_numericos = df["Review and rating"].dropna()
no_numericos = no_numericos[~no_numericos.str.match(r"^\d")]

print("Valores no numéricos encontrados:")
print(no_numericos.value_counts())

## La solución: `pd.to_numeric` con `errors="coerce"`

`.astype(float)` es **intolerante**: si un solo valor falla, se cae todo.

`pd.to_numeric(..., errors="coerce")` es **tolerante**: convierte lo que puede, y lo que no puede lo
transforma en `NaN` (valor faltante).

La palabra *coerce* significa "forzar". Le estamos diciendo: *"forzá la conversión, y lo que no entre,
marcalo como faltante"*.

In [ ]:
# Ahora sí, con la función tolerante
df["rating"] = pd.to_numeric(
    df["Review and rating"].str.split(" ").str[0],
    errors="coerce"
)

# Y de paso extraemos el número de reseñas que está entre paréntesis
df["n_resenas"] = df["Review and rating"].str.extract(r"\((\d+)\)").astype(float)

print(f"✅ Conversión exitosa")
print(f"   Ratings válidos  : {df['rating'].notna().sum()}")
print(f"   Ratings faltantes: {df['rating'].isna().sum()}  (10 'New' + 2 vacíos originales)")
print()
df[["Review and rating", "rating", "n_resenas"]].head()

## 📋 Resumen: de 2 columnas numéricas a 5

Empezamos con `Price` y `Offer price`. Ahora tenemos además `camas`, `rating` y `n_resenas`,
más una variable categórica útil: `tipo`.

**Ahora sí podemos hacer estadística.**

In [ ]:
df[["Price(in dollar)", "camas", "rating", "n_resenas", "tipo"]].head(8)

## ✍️ TU TURNO 3

Contá **cuántos tipos de propiedad distintos** hay en la columna `tipo`.

💡 *Pista: hay un método que cuenta valores únicos. Empieza con `n` y termina con `unique()`.*

In [ ]:
# TU TURNO 3: cuántos tipos de propiedad distintos hay
# Tu código aquí



## ⭐ Desafío opcional 1

Ya extrajimos `camas`, `tipo`, `rating` y `n_resenas`. Falta uno: **el país**.

Los títulos son del estilo `"Chalet in Skykomish, Washington, US"`. El país es lo que viene **después
de la última coma**.

Extraé una columna `pais` y contá cuántos anuncios hay por país.

💡 *Pista: `.str.split(",")` y después `.str[-1]` para el último pedazo. Ojo con los espacios sobrantes:
`.str.strip()` los saca.*

🤔 **Antes de ejecutar:** en la Sección 0 dijimos que 483 de 538 títulos (el 90%) **no** tienen las
tres partes que este método necesita. ¿Cuántos países válidos esperás obtener?


In [ ]:
# ⭐ DESAFÍO 1: extraer el país desde el título
# Tu código aquí




---

# 4. Medidas de tendencia central

## 👨‍🏫 La idea

Tenemos 538 precios. Nadie puede mirar 538 números y sacar una conclusión.

Necesitamos **un solo número que represente al conjunto**. Eso es una medida de tendencia central:
el intento de responder *"¿alrededor de qué valor están estos datos?"*.

Hay tres candidatos, y elegir mal cambia completamente la respuesta.

| Medida | Qué es | Analogía |
|---|---|---|
| **Media** | Suma de todo dividido la cantidad | El punto de equilibrio de un subibaja |
| **Mediana** | El valor del medio al ordenar de menor a mayor | El que está justo en la mitad de la fila |
| **Moda** | El valor que más se repite | El más popular |

Calculemos las tres.

In [ ]:
precio = df["Price(in dollar)"]

media   = precio.mean()
mediana = precio.median()
moda    = precio.mode()[0]   # mode() puede devolver varios; [0] toma el primero

print(f"Media   : ${media:.2f}")
print(f"Mediana : ${mediana:.2f}")
print(f"Moda    : ${moda:.2f}")

## 🤨 Un momento. ¿Cuál es el precio de un Airbnb entonces?

- Si te digo **$175**, no estoy mintiendo.
- Si te digo **$138**, tampoco estoy mintiendo.

Son **$37 de diferencia**. La media es un **27% más alta** que la mediana.

Si estuvieras armando el presupuesto de un viaje, esos $37 por noche × 7 noches = **$259 de diferencia**.
Eso es un pasaje.

**¿Cuál de los dos deberíamos reportar?** Para contestarlo, hay que entender por qué se separan.

## 🎬 El experimento del penthouse fantasma

Vamos a hacer algo que parece una tontería y es la demostración más importante del día.

Imaginemos que aparece **un solo anuncio más**: un penthouse en Dubái a **$50.000 la noche**.
Un anuncio. En 538. Menos del 0.2% del dataset.

🤔 **Predicción — escribí en el chat antes de ejecutar:**
1. ¿Cuánto creés que va a subir la **media**?
2. ¿Cuánto creés que va a subir la **mediana**?

In [ ]:
# Agregamos UN solo anuncio de $50.000
precio_con_penthouse = pd.concat(
    [precio, pd.Series([50000])],
    ignore_index=True
)

print("                    ANTES        DESPUÉS      CAMBIO")
print("                 ---------    ----------   ----------")
print(f"Media            ${precio.mean():>8.2f}    ${precio_con_penthouse.mean():>8.2f}    ${precio_con_penthouse.mean() - precio.mean():>+8.2f}")
print(f"Mediana          ${precio.median():>8.2f}    ${precio_con_penthouse.median():>8.2f}    ${precio_con_penthouse.median() - precio.median():>+8.2f}")

## 💥 El resultado

**Un solo dato movió el promedio $92.44.** Es un aumento del 53%.

**La mediana no se movió ni un centavo.** Cero. Nada. Ni se enteró.

### La analogía para que no se te olvide nunca

Imaginate que estamos todos los de esta clase en una sala y calculamos el **sueldo promedio**.
Da una cifra razonable.

Ahora entra **Marcelo Claure** por la puerta.

- El **sueldo promedio** de la sala pasa a ser de varios millones de dólares.
- Técnicamente cierto. Estadísticamente correcto. **Y completamente inútil** para describir a quienes
  estamos en la sala.
- La **mediana** se movería un puestito. Seguiría diciendo la verdad sobre el grupo.

> ## 💡 Frase para memorizar
> ### La media es sensible a los valores extremos. La mediana es robusta.

## ¿Y por qué en NUESTROS datos la media es más alta?

Porque el dataset **ya tiene sus propios penthouses**. Miremos los cinco anuncios más caros:


In [ ]:
df.nlargest(5, "Price(in dollar)")[["Title", "Price(in dollar)", "camas", "tipo"]]

$955, $950, $913... contra una mediana de $138.

Estos anuncios **existen y son legítimos** — no son errores, son casas de lujo reales. Pero al ser
tan pocos y tan extremos, **arrastran la media hacia arriba** sin representar a nadie.

## ✅ La respuesta a la pregunta ancla

> ### ¿Cuánto cuesta realmente un Airbnb?
> **Unos $138 la noche** (mediana), no $175 (media).
>
> Cuando los datos tienen valores extremos hacia un lado, **la mediana describe mejor al típico**.

## 📊 El resumen que hace todo de una vez: `.describe()`

`describe()` calcula de golpe: cantidad, media, desviación estándar, mínimo, los tres cuartiles y máximo.
Es el punto de partida de cualquier análisis.

In [ ]:
df[["Price(in dollar)", "camas", "rating", "n_resenas"]].describe()

### 👨‍🏫 Esta tabla ES el temario de la clase

Mirá la columna de `Price(in dollar)` y date cuenta de que **ya entendés casi todo**:

| Fila | Qué significa | ¿Ya lo vimos? |
|---|---|---|
| `count` | Cuántos valores no nulos hay | ✅ Sección 2 |
| `mean` | La media | ✅ Recién |
| `std` | Desviación estándar | 👉 Sección 5 |
| `min` / `max` | El más barato y el más caro | ✅ Obvio |
| `25%` `50%` `75%` | Los cuartiles (el 50% **es** la mediana) | 👉 Sección 6 |

**Truco profesional:** comparar `mean` con `50%` de un vistazo. Si son muy distintos, hay asimetría.
Acá: 175 vs 138. **Asimetría confirmada sin hacer un solo gráfico.**

## ✍️ TU TURNO 4

Calculá la **media y la mediana de la columna `rating`**. Después compará: ¿se parecen o son muy distintas?
¿Qué te dice eso sobre la forma de esa variable?

💡 *Pista: mismos métodos que usamos con `precio`.*

In [ ]:
# TU TURNO 4: media y mediana del rating
# Tu código aquí



---

# 5. Medidas de dispersión

## 👨‍🏫 Por qué el centro no alcanza

Dos ciudades pueden tener el **mismo precio promedio** de $150 y ser experiencias completamente distintas:

- **Ciudad A:** todos los anuncios cuestan entre $140 y $160. Predecible.
- **Ciudad B:** la mitad cuesta $30 y la otra mitad $270. Ruleta.

Mismo promedio. **Realidad opuesta.** Por eso un promedio sin una medida de dispersión al lado es
información incompleta — y en un informe profesional, es directamente un error.

Las medidas de dispersión responden: **¿qué tan desparramados están los datos?**

## Medida 1: el rango

La más simple: máximo menos mínimo.

In [ ]:
rango = precio.max() - precio.min()

print(f"Precio mínimo : ${precio.min()}")
print(f"Precio máximo : ${precio.max()}")
print(f"RANGO         : ${rango}")

**$939 de rango.** El anuncio más caro cuesta **60 veces** lo que el más barato.

⚠️ **Debilidad del rango:** usa solamente 2 de los 538 datos. Si esos dos son raros, el rango miente.
Es un indicador rápido, no confiable.

## Medida 2: la varianza

La varianza mide, en promedio, **qué tan lejos está cada dato de la media**.

La idea intuitiva:
1. Para cada precio, calculo su distancia a la media.
2. Elevo cada distancia al cuadrado (para que las de abajo no cancelen a las de arriba).
3. Promedio todo.

In [ ]:
varianza = precio.var()
print(f"Varianza: {varianza:,.2f}")

## 😵 ¿18.593 qué?

Ese número no significa nada intuitivo, y hay una razón: **está en dólares al cuadrado**.

Elevamos las distancias al cuadrado en el paso 2, así que las unidades también se elevaron al cuadrado.
"Dólares cuadrados" no es una unidad que exista en el mundo real.

**Por eso la varianza casi nunca se reporta directamente.** Es un paso intermedio.

## Medida 3: la desviación estándar (la que sí se usa)

Si el problema es que la varianza está al cuadrado... **le sacamos la raíz cuadrada.** Y volvemos a dólares.

In [ ]:
desviacion = precio.std()

print(f"Varianza            : {varianza:,.2f}  (dólares²  🤷)")
print(f"Raíz de la varianza : {np.sqrt(varianza):,.2f}  (dólares  ✅)")
print(f"Desviación estándar : {desviacion:,.2f}  (dólares  ✅)")
print()
print("👆 Las dos últimas son el mismo número. Esa ES la relación entre ambas:")
print("   desviación estándar = √varianza")

## 👨‍🏫 Cómo interpretar $136.36

> **En promedio, los precios se alejan unos $136 de la media.**

Y ahora una comparación que vale oro:

- Media: **$175**
- Desviación estándar: **$136**

**La dispersión es casi tan grande como el propio promedio.** Eso significa que decir "un Airbnb cuesta
$175" es una afirmación con muchísima incertidumbre. Los datos están enormemente desparramados.

## 🎭 El contraejemplo perfecto: la columna `rating`

Ahora miremos una variable que se comporta **exactamente al revés**.

In [ ]:
print("PRECIO")
print(f"  Media               : {precio.mean():.2f}")
print(f"  Desviación estándar : {precio.std():.2f}")
print(f"  Mínimo / Máximo     : {precio.min()} / {precio.max()}")
print()
print("RATING")
print(f"  Media               : {df['rating'].mean():.3f}")
print(f"  Desviación estándar : {df['rating'].std():.3f}")
print(f"  Mínimo / Máximo     : {df['rating'].min()} / {df['rating'].max()}")

## 🌟 Casi todos tienen 4.9 estrellas

La desviación estándar del rating es **0.137**. Prácticamente cero.

Para dimensionarlo: el **50% central** de los anuncios calificados está entre **4.81 y 4.96** — un
rango de apenas 15 centésimas. Y **ninguno de los 526 baja de 4.0**.

> 📌 Ojo con exagerar el dato: no es que *todos* estén entre 4.8 y 5.0. Alrededor de **un 22% queda
> por debajo de 4.8**. Pero incluso ese "22% peor calificado" vive entre 4.0 y 4.8, que en cualquier
> otra escala serían notas excelentes. **El argumento se sostiene sin necesidad de estirarlo**, y
> estirarlo es exactamente el tipo de cosa que después alguien te discute en una reunión.

### La reflexión

En Airbnb pasa lo mismo que en los cumpleaños: **casi nadie le pone menos de 4 estrellas a nadie**. Te
quedaste sin agua caliente, había una cucaracha, el anfitrión te habló raro... y ponés 4 estrellas
"para no arruinarle el negocio al pobre señor".

*(Dicho con rigor: eso es lo que pasa **en estos 538 anuncios**. Que el mínimo sea exactamente 4.0
también podría ser un artefacto de cómo se armó el dataset — lo cual refuerza el punto de la Sección 0:
no sabemos cómo se eligieron.)*

### Y la consecuencia estadística, que es lo importante

> ## 💡 Una variable sin dispersión no discrimina.
> ### Y una variable que no discrimina, no informa.

Si vas a elegir entre dos Airbnb y uno tiene 4.87 y el otro 4.91, **esa diferencia no te dice nada**:
está dentro del ruido normal de la variable. El rating de Airbnb, estadísticamente, es casi inútil
para comparar.

El precio, en cambio, con su desviación de $136, **sí discrimina muchísimo**.

### 💼 Y esto, ¿para qué te sirve en la práctica?

Esta es una de las preguntas más útiles que podés hacerle a cualquier variable en tu trabajo:

> **¿Esta variable varía lo suficiente como para servirme de algo?**

Aplicaciones directas según tu área:
- **Sistemas:** si todos tus servidores responden entre 48 y 52 ms, dejá de optimizar latencia y mirá otra cosa.
- **Finanzas / auditoría:** un control cuyo indicador da siempre lo mismo no está detectando nada; es un control decorativo.
- **RRHH:** si todas las evaluaciones de desempeño dan entre 4.7 y 5.0, el instrumento no sirve para decidir ascensos.

**Antes de construir un modelo, un tablero o un KPI con una variable, mirá su dispersión.** Si no
varía, no va a explicar nada — y te ahorrás el trabajo de descubrirlo al final.

## ✍️ TU TURNO 5

Calculá la desviación estándar de `n_resenas` (número de reseñas) y compará con su media.

¿La cantidad de reseñas está muy dispersa o poco dispersa? ¿Se parece más al precio o al rating?

💡 *Pista: `.mean()` y `.std()` sobre `df["n_resenas"]`.*


In [ ]:
# TU TURNO 5: dispersión del número de reseñas
# Tu código aquí



## ⭐ Desafío opcional 2 — el coeficiente de variación

Tenemos un problema: **no se puede comparar una desviación de $136 con una de 0.137 estrellas.**
Son unidades distintas. Es como preguntar si tres kilos son más que dos metros.

La solución se llama **coeficiente de variación (CV)**: la desviación estándar expresada como
**porcentaje de la media**.

$$CV = \frac{s}{\bar{x}} \times 100$$

Como es un porcentaje, **es adimensional** — y por eso sí se puede comparar entre variables.

Calculá el CV de las cuatro variables numéricas (`Price(in dollar)`, `n_resenas`, `camas`, `rating`)
y ordenalas de más a menos variable.

💡 *Pista: `(serie.std() / serie.mean()) * 100`. Podés hacer un `for` sobre una lista de nombres de columna.*

🤔 **Predicción:** ¿cuál creés que va a salir primera y cuál última?


In [ ]:
# ⭐ DESAFÍO 2: coeficiente de variación de las cuatro variables numéricas
# Tu código aquí




---

# 6. Medidas de posición y forma de la distribución

## Percentiles y cuartiles

Un **percentil** responde: *"¿qué valor deja por debajo al X% de los datos?"*

Los tres más usados se llaman **cuartiles** porque parten los datos en cuatro grupos iguales:

| Cuartil | Percentil | Significado |
|---|---|---|
| **Q1** | 25% | El 25% más barato está por debajo |
| **Q2** | 50% | La **mediana** |
| **Q3** | 75% | El 75% está por debajo (el 25% más caro, arriba) |

In [ ]:
cuartiles = precio.quantile([0.25, 0.50, 0.75])
print(cuartiles)

## 👨‍🏫 Cómo se lee esto en lenguaje humano

- **Q1 = $90** → 1 de cada 4 anuncios cuesta menos de $90.
- **Q2 = $138** → la mitad cuesta menos de $138 (es la mediana, ya la conocíamos).
- **Q3 = $222** → 3 de cada 4 cuestan menos de $222. Solo el 25% supera esa cifra.

**Ahora sí podés hablar como un analista:**

> *"El rango típico de precios va de $90 a $222 la noche, con una mediana de $138."*

Esa frase describe el mercado infinitamente mejor que "el promedio es $175".

## Percentiles personalizados

No estás limitado a los cuartiles. Podés pedir cualquiera.

In [ ]:
df["Price(in dollar)"].quantile([0.10, 0.50, 0.90, 0.99])

### 👨‍🏫 Cómo se leen los percentiles personalizados

- **P10 = $52.70** → el 10% más barato del mercado está por debajo de ~$53 la noche.
- **P50 = $138** → la mediana, ya la conocíamos.
- **P90 = $322** → solo 1 de cada 10 anuncios supera los $322.
- **P99 = $728.30** → si un anuncio pasa los ~$728, está en el **1% más caro** del dataset.

### 💼 Para qué sirve esto en la práctica

Los percentiles son la forma profesional de fijar umbrales **sin inventarlos**:

- **Sistemas:** el famoso **p95 de latencia** de los acuerdos de nivel de servicio. "El 95% de las
  peticiones responde en menos de X ms" es un compromiso medible; "el promedio es X ms" no lo es.
- **Comercial:** definir el segmento *premium* como "todo lo que supere el P90" en vez de elegir un
  número redondo a dedo.
- **Auditoría:** marcar para revisión las transacciones por encima del P99, que es exactamente el 1%
  más inusual.

> 💡 **La diferencia clave:** un umbral fijado con un percentil **se adapta solo** cuando cambian los
> datos. Un umbral fijado a mano ("todo lo que supere $700") queda viejo y nadie se entera.

## 📊 El histograma: ver la forma

### 👨‍🏫 Qué es y para qué sirve

Un **histograma** divide el rango de una variable numérica en intervalos consecutivos (*bins*) y
dibuja una barra cuya altura es **cuántos datos caen en cada intervalo**. Dicho simple: **es el censo
de tus datos** — los organiza en cajones por tamaño y te muestra qué cajón está más lleno.

**Para qué sirve en la vida real:** ver la estructura salarial real de una empresa, la distribución
de tiempos de respuesta de una API, los montos de transacción para calibrar alertas, la distribución
de notas de un curso. **Su uso estrella:** detectar que tenés **dos poblaciones mezcladas** — si el
histograma tiene dos jorobas, casi siempre estás promediando dos cosas distintas.

**Cómo se lee:** el eje X es la variable (en intervalos), el eje Y es **cuántos casos** hay en cada uno.
Mirá tres cosas: dónde está el pico, si es simétrico o tiene cola, y si hay huecos.

**Cómo se invoca:** `plt.hist(datos, bins=30, edgecolor="black")`.

> ⚠️ **Diferencia con el gráfico de barras:** en un histograma las barras **se tocan**, porque el eje X
> es continuo (no hay hueco entre $100 y $101). En un gráfico de barras van separadas, porque las
> categorías son independientes.

> 💡 **¿Cuántos bins?** No hay respuesta única, y ahí está la trampa: el mismo dataset parece una
> campana suave o un peine dentado según cuántos elijas. Regla práctica: **la raíz cuadrada de la
> cantidad de datos** (√538 ≈ 23; nosotros usamos 30, que está en rango razonable). Si alguien te
> muestra un histograma con un número raro de bins, preguntale por qué eligió ese.


In [ ]:
plt.figure(figsize=(10, 5))

plt.hist(precio, bins=30, edgecolor="black", color="#4C9BE8")

# Marcamos media y mediana para verlas en contexto
plt.axvline(media,   color="red",   linestyle="--", linewidth=2, label=f"Media = ${media:.0f}")
plt.axvline(mediana, color="green", linestyle="--", linewidth=2, label=f"Mediana = ${mediana:.0f}")

plt.title("Distribución de precios de Airbnb", fontsize=14, fontweight="bold")
plt.xlabel("Precio por noche (USD)")
plt.ylabel("Cantidad de anuncios")
plt.legend()
plt.show()

## 👨‍🏫 Ahí está. Esa es la explicación visual de toda la clase.

Mirá la forma: una **montaña a la izquierda** y una **cola larga que se estira a la derecha**.

Y mirá dónde quedaron las líneas:
- La **verde (mediana)** está clavada en el pico, donde está la mayoría de los anuncios.
- La **roja (media)** está corrida a la derecha, **arrastrada por la cola**.

Esto se llama **asimetría positiva** o **sesgo a la derecha**. Pandas incluso lo mide:

In [ ]:
asimetria = precio.skew()
print(f"Asimetría (skewness): {asimetria:.2f}")
print()
if asimetria > 1:
    print("👉 Asimetría positiva FUERTE: cola larga hacia la derecha.")
    print("   La media está inflada por los valores altos.")
    print("   👉 En este caso se reporta la MEDIANA.")

### La regla práctica que te vas a llevar

| Si... | Entonces la distribución... | Reportá... |
|---|---|---|
| media ≈ mediana | es simétrica | media (o cualquiera) |
| **media > mediana** | **tiene cola a la derecha** | **mediana** |
| media < mediana | tiene cola a la izquierda | mediana |

### 💼 Para qué te sirve esto en la práctica

Es una **decisión de reporte**, y se toma todos los días:

- **Sueldos, alquileres, ingresos, precios de vivienda, tiempos de respuesta**: son notoriamente
  asimétricos. Por eso los organismos oficiales publican el **ingreso mediano**, no el promedio.
- Si tu jefe te pide "el ticket promedio" y la distribución tiene cola, **dale los dos números y
  explicá la diferencia**. Es lo que separa a un analista de alguien que ejecuta funciones.

## 📦 El boxplot: cinco estadísticos en un solo dibujo

### 👨‍🏫 Qué es y para qué sirve

El **boxplot** (diagrama de caja, o *caja y bigotes*) resume una variable numérica con **cinco
números**: mínimo, Q1, mediana, Q3 y máximo — más los valores atípicos dibujados aparte.

**Su virtud:** densidad de información. En un dibujo del tamaño de una estampilla te dice dónde está
el centro, cuán dispersos están los datos, si hay asimetría y quiénes son los raros.
**Su defecto:** hay que **aprender a leerlo**, no es intuitivo la primera vez.

**Para qué sirve en la vida real:** comparar sucursales o vendedores, evaluar proveedores (quién es
rápido *y* quién es consistente), detectar fraude (los puntos fuera de los bigotes son los candidatos
a revisar), comparar tratamientos en investigación. **Su superpoder es la comparación**: entran ocho
boxplots en el ancho de una diapositiva; ocho histogramas, no.

**Cómo se invoca:** `plt.boxplot(datos, vert=False)` — `vert=False` lo pone horizontal, que para una
sola variable se lee mejor.


In [ ]:
plt.figure(figsize=(10, 4))

plt.boxplot(precio, vert=False, widths=0.6)

plt.title("Boxplot de precios de Airbnb", fontsize=14, fontweight="bold")
plt.xlabel("Precio por noche (USD)")
plt.yticks([])
plt.grid(axis="x", alpha=0.3)
plt.show()

## 👨‍🏫 Cómo leer un boxplot

Ese dibujo aparentemente simple contiene **cinco números**:

```
        Q1      Q2      Q3
         │       │       │
   ├─────┤███████│███████├────────┤    o   o  o   o
   │     └───────┴───────┘        │
 bigote        LA CAJA          bigote      outliers
 inferior                       superior
```

| Elemento | Qué es | Valor acá |
|---|---|---|
| Borde izquierdo de la caja | Q1 | $90 |
| Línea dentro de la caja | **Mediana** | $138 |
| Borde derecho de la caja | Q3 | $222 |
| Ancho de la caja | **IQR** (rango intercuartílico) | $132 |
| Puntos sueltos a la derecha | **Outliers** | 28 anuncios |

**La caja contiene exactamente al 50% central de los datos.** Los puntos sueltos son los valores
atípicos: tan lejos del grupo que el gráfico los dibuja aparte.

## ¿Cómo decide qué es un outlier?

Hay una regla estándar. Se calcula el IQR (Q3 − Q1) y se marca como outlier todo lo que esté más allá
de **1.5 × IQR** de los bordes de la caja.

In [ ]:
Q1 = precio.quantile(0.25)
Q3 = precio.quantile(0.75)
IQR = Q3 - Q1

limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

print(f"Q1  = ${Q1:.2f}")
print(f"Q3  = ${Q3:.2f}")
print(f"IQR = ${IQR:.2f}   (Q3 - Q1)")
print()
print(f"Límite inferior = ${limite_inferior:.2f}")
print(f"Límite superior = ${limite_superior:.2f}")
print()

outliers = df[df["Price(in dollar)"] > limite_superior]
print(f"🔍 Anuncios considerados outliers: {len(outliers)}")
print(f"   Eso es el {len(outliers) / len(df) * 100:.1f}% del dataset")

## ⚠️ Advertencia importante sobre los outliers

**Outlier NO significa error.**

Esos 28 anuncios de más de $420 son villas y casas de lujo **perfectamente reales**. No hay nada
que corregir.

Un outlier es simplemente un dato **muy distinto al resto**. Puede ser:

| Un outlier puede ser... | Qué hacer |
|---|---|
| Un error de carga | Corregirlo o eliminarlo, **documentándolo** |
| Un caso legítimo excepcional | **Conservarlo** y reportarlo aparte |
| **El dato más importante del análisis** | En fraude, seguridad y auditoría, **es justamente lo que buscás** |

> 💡 Borrar outliers "porque afean el gráfico" es una de las formas más comunes de mentir con datos.
> Si los sacás, tenés que decirlo y justificarlo.

### 🎓 Dos detalles finos que valen oro

**El límite inferior dio negativo (−$108).** Un alojamiento a menos ciento ocho dólares la noche te
pagaría por ir. Como eso no existe, **no hay ni un solo outlier del lado barato**: los 28 están todos
del lado caro. Esa asimetría en los outliers es otra forma de ver el sesgo positivo.

**El bigote del boxplot no termina en $420, termina en $417.** El límite es $420, pero el bigote se
dibuja hasta **el último dato real que queda dentro del límite**. No hay ningún anuncio de exactamente
$420; el más caro que no lo supera cuesta $417. **El límite es el criterio; el bigote toca un dato real.**

### 💼 El valor de negocio de este ejercicio

Acabás de segmentar el mercado sin que nadie te dijera cómo: hay un **mercado masivo** (de $16 a $420,
el 94.8% de los anuncios) y un **mercado de lujo** (28 anuncios, el 5.2%). Son dos negocios distintos,
con clientes distintos y precios que no se comparan entre sí.

Si fueras a publicar tu propio Airbnb, el precio de referencia sale del primer grupo, **no del promedio
global de $175** — que está contaminado por un segmento en el que no vas a competir.

## 🔔 Bonus: ¿el precio sigue una distribución normal?

En el notebook oficial de Khipus vas a ver un gráfico que dibuja una **campana de Gauss** (distribución
normal) usando la media y la desviación estándar del precio.

Vamos a reproducirlo y a hacernos una pregunta incómoda.


In [ ]:
# La campana normal teórica que tendría estos mismos media y desviación
x = np.linspace(precio.min(), precio.max(), 500)
y_normal = (1 / (desviacion * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((x - media) / desviacion) ** 2)

plt.figure(figsize=(11, 5))

# Histograma real, normalizado para poder compararlo con la curva
plt.hist(precio, bins=40, density=True, alpha=0.6,
         color="#4C9BE8", edgecolor="black", label="Datos REALES")

# La campana teórica
plt.plot(x, y_normal, color="red", linewidth=2.5,
         label="Distribución normal teórica")

plt.axvline(media, color="darkgreen", linestyle="--", label=f"Media = ${media:.0f}")

plt.title("¿Los precios siguen una distribución normal?", fontsize=14, fontweight="bold")
plt.xlabel("Precio por noche (USD)")
plt.ylabel("Densidad")
plt.legend()
plt.show()

## 🤔 Pregunta para el chat: ¿se parecen?

**No.** Y no se parecen en nada.

- La campana roja es **simétrica**; los datos azules están **volcados a la izquierda**.
- La campana predice **precios negativos** (mirá cómo se extiende hacia la izquierda del cero).
  Un Airbnb a −$50 la noche te pagaría por dormir ahí. No existe.
- La campana subestima cuántos anuncios baratos hay y no captura la cola larga.

### 👨‍🏫 Por qué esto importa

Este gráfico aparece en el material oficial y es útil **para entender qué es la desviación estándar**.
Pero como descripción de nuestros datos, **es incorrecto**: el precio no es normal, tiene asimetría 2.34.

Y acá está el puente hacia lo que viene:

> La **estadística inferencial** (nuestro próximo módulo) apoya muchísimas de sus técnicas en el
> supuesto de normalidad. Si aplicás esas técnicas a datos como estos sin verificar la forma primero,
> las conclusiones salen mal.
>
> **Verificar la forma de la distribución antes de elegir el método NO es opcional.**

## ✍️ TU TURNO 6

Hacé un **histograma de la columna `rating`** con 20 bins.

Antes de ejecutarlo: 🤔 ¿qué forma esperás que tenga, sabiendo que la desviación estándar es 0.137?

💡 *Pista: copiá la estructura del histograma de precios, cambiando la columna.*

In [ ]:
# TU TURNO 6: histograma del rating
# Tu código aquí



---

# 7. Variables categóricas: frecuencias

## 👨‍🏫 Un cambio de reglas

Todo lo anterior (media, mediana, desviación, cuartiles) funciona con **variables numéricas**.

Con variables **categóricas** —como `tipo`— nada de eso aplica. No existe "el tipo de propiedad promedio".
No podés decir que el promedio entre `Cabin` y `Villa` es `Cabilla`.

Para categóricas hay otras herramientas:

| Herramienta | Qué hace |
|---|---|
| **Tabla de frecuencias** | Cuenta cuántas veces aparece cada categoría |
| **Frecuencia relativa** | Lo mismo, pero en porcentaje |
| **Moda** | La categoría más frecuente (la única medida de centro que aplica) |

## Tabla de frecuencias: `value_counts()`

In [ ]:
frecuencias = df["Number of bed"].value_counts()
frecuencias

## Frecuencia relativa (porcentajes)

Los números absolutos engañan si no sabés el total. `normalize=True` los convierte en proporciones.

In [ ]:
frecuencia_relativa = (df["Number of bed"].value_counts(normalize=True) * 100).round(1)
print("Porcentaje de anuncios según cantidad de camas:")
print(frecuencia_relativa)

**El 41% de los anuncios tiene 2 camas.** Esa es la **moda** de esta variable.

Sumando 1, 2 y 3 camas ya cubrimos más del 80% del mercado: **Airbnb es mayoritariamente un negocio
de alojamientos chicos.**

## Gráfico de barras

In [ ]:
plt.figure(figsize=(9, 5))

frecuencias.plot(kind="bar", color="#4C9BE8", edgecolor="black")

plt.title("Cantidad de anuncios según número de camas", fontsize=14, fontweight="bold")
plt.xlabel("Número de camas")
plt.ylabel("Cantidad de anuncios")
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.3)
plt.show()

## 🥧 El gráfico de torta (y por qué los estadísticos lo detestan)

### 👨‍🏫 Qué es y cómo se lee

El **gráfico de torta** (*pie chart*) representa la composición de un total dividiendo un círculo en
sectores, donde **el ángulo de cada sector** es proporcional a la fracción que representa (360° = 100%).

**Cómo se lee:** cada porción es una categoría; su tamaño angular es su peso sobre el total. Se lee
bien cuando hay **pocas categorías de tamaños muy distintos**, y mal en cualquier otro caso.

**Cómo se invoca:** `frecuencias.plot(kind="pie", autopct="%1.1f%%", startangle=90)`.
`autopct` escribe los porcentajes sobre cada sector; `startangle=90` arranca a las 12 en punto,
que se lee mejor.

El pie chart aparece en el notebook oficial, se pide en todas las oficinas del mundo, y tu jefe te lo
va a pedir. Así que hay que saber hacerlo — **y saber por qué conviene no hacerlo.**


In [ ]:
plt.figure(figsize=(7, 7))

frecuencias.plot(kind="pie", autopct="%1.1f%%", startangle=90)

plt.title("Distribución del número de camas", fontsize=14, fontweight="bold")
plt.ylabel("")   # pandas pone un label feo por defecto
plt.show()

## 👨‍🏫 Ahora la crítica honesta

Mirá el gráfico de torta y respondé rápido: **¿"4 beds" es más grande o más chico que "1 bed"?**

Te costó, ¿no? Tuviste que leer los porcentajes.

Ahora volvé al **gráfico de barras** de arriba y respondé la misma pregunta. Instantáneo.

### La evidencia, en grados

Estos son los ángulos reales de nuestro gráfico:

| Categoría | % | Grados |
|---|---:|---:|
| 2 beds | 41.1% | 147.9° |
| 3 beds | 21.7% | 78.3° |
| 1 bed | 18.6% | 66.9° |
| 4 beds | 11.7% | 42.2° |
| 5 beds | 3.2% | **11.4°** |
| 6 beds | 3.2% | **11.4°** |
| 7 beds | 0.6% | 2.0° |

Fijate en dos cosas:

- **"5 beds" y "6 beds" tienen exactamente el mismo ángulo (11.4°)** porque ambos tienen 17 anuncios.
  Pero desde el gráfico **no hay forma de saber si son iguales o solo parecidos**.
- **"3 beds" contra "1 bed": 11 grados de diferencia.** Son 117 anuncios contra 100 — una diferencia
  real del 17% que en barras se ve al instante y en la torta hay que leerla.

> **Si tenés que leer los números para entender el gráfico, el gráfico no está funcionando.**

### El motivo técnico

El ojo humano **compara longitudes muy bien y ángulos muy mal**. Un gráfico de barras codifica la
información en longitud; la torta la codifica en ángulo. Esto no es opinión: viene de los trabajos de
**Cleveland y McGill (1984)** sobre percepción gráfica, que ordenaron experimentalmente qué tan bien
percibimos cada tipo de codificación — **posición y longitud arriba, ángulo bastante más abajo**.

| Situación | Gráfico recomendado |
|---|---|
| Comparar categorías entre sí | **Barras** ✅ |
| Más de 5 categorías | **Barras** (la torta se vuelve ilegible) |
| Mostrar partes de un todo, 2 o 3 categorías | Torta, aceptable |
| Mostrar evolución en el tiempo | **Nunca** torta. Va gráfico de líneas |

> 💡 **Consejo profesional:** hacé el gráfico de torta si te lo piden — pero poné el de barras al lado.
> Suele ganar la discusión solo, y ganar un argumento sin discutir es la mejor forma de ganarlo.

## ✍️ TU TURNO 7

Mostrá la tabla de frecuencias de la columna **`tipo`**, pero solo los **10 tipos más comunes**.

💡 *Pista: `value_counts()` ya devuelve ordenado de mayor a menor. Solo necesitás quedarte con los primeros 10.*


In [ ]:
# TU TURNO 7: los 10 tipos de propiedad más frecuentes
# Tu código aquí



---

# 8. Comparar grupos: `groupby()`

## 👨‍🏫 La pregunta que sigue naturalmente

Ya sabemos que un Airbnb cuesta unos $138 la noche (mediana). Pero eso mezcla habitaciones compartidas
con villas de lujo.

La pregunta real es: **¿cuánto cuesta cada tipo de alojamiento?**

`groupby()` parte los datos en grupos y calcula estadísticos para cada uno por separado. Es,
probablemente, **la función más poderosa de pandas**.

## Precio por tipo de propiedad

In [ ]:
# Nos quedamos con los 8 tipos más frecuentes (los demás tienen muy pocos casos)
tipos_frecuentes = df["tipo"].value_counts().head(8).index

resumen_por_tipo = (
    df[df["tipo"].isin(tipos_frecuentes)]
    .groupby("tipo")["Price(in dollar)"]
    .agg(["count", "mean", "median", "std"])
    .round(1)
    .sort_values("median")
)

resumen_por_tipo

## 👨‍🏫 Cómo leer una tabla de `groupby`

**Regla absoluta: la primera columna que se lee es siempre `count`.**

Antes de emocionarte con una media, mirá cuántos datos la sostienen. Una media calculada sobre 3
observaciones no es un resultado: es una anécdota con decimales.

Ahora sí, la historia que cuenta la tabla:

| Tipo | Mediana | Traducción |
|---|---|---|
| **Room** | $100.5 | Una habitación en casa ajena. El piso del mercado. |
| **Apartment** | $117 | El estándar urbano. |
| **Condo** | $150 | Departamento con amenities. |
| **Cabin** | $171 | Ya estás pagando por la experiencia. |
| **Home** | $192 | Casa entera. |
| **Villa** | $222 | El techo. Más del doble que una habitación. |

**Una villa cuesta 2.2 veces lo que una habitación.** Eso es información accionable — y era invisible
cuando mirábamos un solo promedio global.

### El segundo hallazgo, que casi nadie mira: la columna `std`

`Room` tiene desviación **66** y `Treehouse` tiene **210**. Las habitaciones son un mercado
**predecible**; las casas del árbol son una **lotería**.

Y si mirás la tabla completa, la dispersión **crece a medida que sube el precio**. Eso tiene nombre
técnico —**heterocedasticidad**, que suena a enfermedad respiratoria pero solo significa "dispersión
no constante"— y una explicación de negocio simple:

> Una habitación es una habitación: cuatro paredes, una cama, quizás baño privado. El precio está
> acotado por arriba. Una casa puede ser una casita modesta o una mansión con piscina infinita.
> **El techo del lujo no existe.**

### 💼 Qué decisión sale de acá

Esta tabla es, literalmente, un **estudio de mercado**:

- **Si vas a publicar una propiedad:** ya sabés en qué franja te tenés que ubicar según su tipo, y
  cuánta variación es normal en tu segmento.
- **Si vas a invertir:** `Tiny home` tiene el precio más predecible (desviación 48) y `Treehouse` el
  más volátil (210), seguido de `Home` (187). Uno es un negocio de flujo estable; los otros, de alto
  riesgo y alto techo.
- **La lección transferible:** *"el promedio global de $175"* no era falso, era **inútil**. No describía
  a ningún segmento real. Casi siempre que un promedio se siente raro, es porque **hay grupos adentro
  que no se separaron**.

## 📦 Boxplot comparativo

### 👨‍🏫 Cómo se lee

Es el mismo boxplot de antes, pero **uno por grupo, alineados sobre el mismo eje Y**. Ese eje
compartido es lo que hace que la comparación funcione — es la razón de ser del gráfico.

Se lee de dos formas a la vez: **la altura de las cajas** compara los centros (qué segmento es más caro)
y **el tamaño de las cajas** compara las dispersiones (qué segmento es más predecible).

> ⚠️ **Detalle importante:** cada caja calcula sus outliers con **su propio IQR**, no con el del
> dataset completo. Un anuncio de $400 puede ser outlier dentro de "Room" y perfectamente normal
> dentro de "Villa".

> ⚠️ **Lo que este gráfico NO muestra: cuántos datos tiene cada caja.** La de `Treehouse` (20 casos)
> se ve exactamente igual de sólida que la de `Home` (83). Si lo presentás, **poné el `n` debajo de
> cada caja.**


In [ ]:
# matplotlib necesita una lista de arrays, uno por grupo
tipos_ordenados = resumen_por_tipo.index.tolist()
datos_por_tipo = [
    df[df["tipo"] == t]["Price(in dollar)"].values
    for t in tipos_ordenados
]

plt.figure(figsize=(11, 6))
plt.boxplot(datos_por_tipo, vert=True)

plt.title("Distribución de precios según tipo de propiedad", fontsize=14, fontweight="bold")
plt.xlabel("Tipo de propiedad")
plt.ylabel("Precio por noche (USD)")
# Los boxplots se numeran 1, 2, 3... así que les ponemos el nombre de cada tipo
plt.xticks(range(1, len(tipos_ordenados) + 1), tipos_ordenados, rotation=30, ha="right")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

### 👨‍🏫 Nota técnica

Fijate lo verboso que fue armar ese gráfico: hubo que construir a mano una lista con los datos de cada
grupo. Con **seaborn** habría sido una sola línea (`sns.boxplot(data=df, x="tipo", y="precio")`).

Esa es exactamente la razón por la que seaborn existe. No la usamos hoy para mantener consistencia con
el material oficial, pero **sabé que existe y que te va a ahorrar mucho tiempo** más adelante.

## 🎬 La sorpresa: ¿más camas = más caro?

Parece obvio que sí. Una casa de 6 camas debería costar más que un monoambiente.

🤔 **Predicción antes de ejecutar:** escribí en el chat si creés que el precio **sube siempre** a medida
que aumentan las camas.

In [ ]:
precio_por_camas = (
    df.groupby("camas")["Price(in dollar)"]
    .agg(["count", "mean", "median"])
    .round(1)
)

precio_por_camas

## 😲 El dato desmiente la intuición

Mirá la columna `median` bajando por la tabla:

| Camas | Mediana |
|---:|---:|
| 1 | $118 |
| 2 | $142 |
| 3 | $138 ⬇️ |
| 4 | $136 ⬇️ |
| 5 | $192 |
| 6 | $250 |
| 7 | $220 ⬇️ |

**De 2 a 4 camas el precio BAJA.** No sube. Y de 6 a 7 vuelve a bajar.

### Pero antes de teorizar... mirá `count`

- 5 camas: **17 anuncios**
- 6 camas: **17 anuncios**
- 7 camas: **solo 3 anuncios**

**Con 3 observaciones no se concluye nada.** Ese $220 podría cambiar completamente si aparecieran dos
anuncios más. Los grupos de 1 a 4 camas (con 100, 221, 117 y 63 casos) sí son confiables; los de 5 a 7,
no tanto.

### 👨‍🏫 La lección doble

1. **Tu intuición sobre los datos puede estar equivocada.** Por eso se miden las cosas en vez de suponerlas.
2. **Un resultado sin tamaño de muestra no es un resultado.** Siempre, siempre mirá `count`.

> 💡 La ubicación, el tipo de propiedad y la temporada pesan más en el precio que la cantidad de camas.
> Una habitación en París cuesta más que una casa de 4 camas en un pueblo.

## ✍️ TU TURNO 8

Usá `groupby` para calcular el **rating promedio por tipo de propiedad**.

¿Hay algún tipo que se destaque? (Acordate de la Sección 5: el rating casi no varía... ¿se mantiene eso
al separar por grupos?)

💡 *Pista: mismo patrón que arriba, cambiando la columna que agregás por `"rating"`.*

In [ ]:
# TU TURNO 8: rating promedio por tipo de propiedad
# Tu código aquí



---

# 9. Relación entre dos variables

## El scatter plot (gráfico de dispersión)

### 👨‍🏫 Qué es y para qué sirve

Hasta ahora miramos variables de a una. Un **scatter plot** muestra **dos variables numéricas a la vez**:
cada punto es una observación, ubicada según su valor en X y en Y.

Es el gráfico fundamental para responder: **¿estas dos cosas se mueven juntas?**

**Para qué sirve en la vida real:** inversión publicitaria vs. ventas, años de experiencia vs. salario,
temperatura del proceso vs. defectos, metros cuadrados vs. precio. **Es el paso previo obligatorio a
cualquier regresión** — que es justamente el tema de la Clase 3 de este módulo.

**Cómo se lee:** buscá la **forma de la nube**. ¿Sube de izquierda a derecha (relación positiva)?
¿Baja (negativa)? ¿No tiene forma (sin relación)? ¿Es una curva? ¿Hay grupos separados? ¿Hay unos
pocos puntos gobernando todo?

> 💡 **Lo que un scatter revela y una correlación no:** el coeficiente es **un solo número**; el gráfico
> te muestra **la forma**. Dos variables pueden tener una relación perfecta en forma de U y dar
> correlación cero. Por eso **siempre se grafica**, no se confía solo en el número.

**Cómo se invoca:** `plt.scatter(x, y, alpha=0.5)`. El `alpha` es transparencia: donde se superponen
puntos se ve más oscuro. Es la mejora más barata y efectiva que existe cuando hay muchos datos encima.

Reproduzcamos el gráfico del notebook oficial: precio contra precio de oferta — pero agregándole una
**línea diagonal de referencia** que el oficial no tiene, y que va a delatar todo.


In [ ]:
plt.figure(figsize=(9, 7))

plt.scatter(df["Price(in dollar)"], df["Offer price(in dollar)"], alpha=0.5, color="#4C9BE8")

# Línea diagonal: acá caerían los anuncios donde oferta == precio
maximo = 1000
plt.plot([0, maximo], [0, maximo], color="red", linestyle="--", linewidth=2,
         label="Oferta = Precio original")

plt.title("Precio original vs. Precio de oferta", fontsize=14, fontweight="bold")
plt.xlabel("Precio original (USD)")
plt.ylabel("Precio de oferta (USD)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 🕵️ Dos cosas que este gráfico esconde

### Escondido #1: ¿cuántos puntos hay dibujados?

Deberían ser 538. **Contemos.**

In [ ]:
puntos_dibujados = df[["Price(in dollar)", "Offer price(in dollar)"]].dropna().shape[0]

print(f"Filas en el dataset        : {len(df)}")
print(f"Puntos realmente dibujados : {puntos_dibujados}")
print(f"Filas que desaparecieron   : {len(df) - puntos_dibujados}  ({(len(df)-puntos_dibujados)/len(df)*100:.1f}%)")

**443 anuncios desaparecieron del gráfico y matplotlib no dijo una palabra.**

Cuando un valor es `NaN`, el punto simplemente no se dibuja. Sin error, sin advertencia, sin nota al pie.

> ## ⚠️ Frase para memorizar
> ### Un gráfico nunca te avisa cuántos datos le faltan. Tenés que preguntárselo vos.

Si presentás este gráfico en una reunión diciendo "así se comportan los precios de oferta", estás
describiendo al 17.7% del dataset y haciéndolo pasar por el total.

**Este es probablemente el hallazgo más transferible de toda la clase.** Vale para todos los gráficos,
todas las herramientas y todos los informes que vayas a ver en tu vida profesional.

### Escondido #2: la línea roja

Todos los puntos **por encima** de la línea roja son anuncios donde la "oferta" es más cara que el
precio original. Contá cuántos hay arriba: son **48 de 95**, aproximadamente la mitad.

Ya lo sabíamos de la Sección 2 — pero ahora **se ve**. Esa es la diferencia entre calcular y visualizar:
**el número te informa, el gráfico te convence.**

Y un detalle que remata la auditoría: **cero puntos caen exactamente sobre la diagonal**. Ni un solo
anuncio tiene precio de oferta idéntico al original, lo cual refuerza la sospecha de que esa columna
se generó de forma poco cuidadosa.

## Correlación

La **correlación** mide qué tan relacionadas están dos variables numéricas, en un número entre −1 y +1.
El coeficiente estándar es el de **Pearson**.

| Valor | Significado |
|---|---|
| **+1** | Relación positiva perfecta: sube una, sube la otra |
| **0** | Sin relación lineal |
| **−1** | Relación negativa perfecta: sube una, baja la otra |

Guía práctica de interpretación:

| \|r\| | Fuerza |
|---|---|
| 0.0 – 0.2 | Nula o muy débil |
| 0.2 – 0.4 | Débil |
| 0.4 – 0.6 | Moderada |
| 0.6 – 0.8 | Fuerte |
| 0.8 – 1.0 | Muy fuerte |

**Cómo se invoca:** `df[["col1", "col2", "col3"]].corr()` devuelve la **matriz** de correlaciones
entre todas las columnas numéricas que le pases.


In [ ]:
matriz_correlacion = df[["Price(in dollar)", "camas", "rating", "n_resenas"]].corr().round(3)
matriz_correlacion

## 👨‍🏫 Cómo leer una matriz de correlación

- La **diagonal siempre es 1**: toda variable está perfectamente correlacionada consigo misma.
- La matriz es **simétrica**: el valor arriba y abajo de la diagonal es el mismo.
- Solo hay que leer **la mitad**.

Miremos la fila del precio:

| Relación | r | Interpretación |
|---|---:|---|
| Precio ↔ rating | 0.197 | Muy débil. Casi nada. |
| Precio ↔ camas | 0.149 | Muy débil. |
| Precio ↔ reseñas | −0.027 | **Prácticamente cero.** |

## 😅 Todas las correlaciones son débiles

Después de todo este trabajo, buscábamos qué explica el precio de un Airbnb... **y no encontramos nada.**

### ¿Fracasamos?

**No.** Esto es un resultado legítimo y hay que saber reportarlo.

Lo que aprendimos es real: **ni la cantidad de camas, ni el rating, ni la popularidad explican el precio**
en este dataset. Lo que probablemente sí lo explica —la ubicación, la temporada, el lujo, los metros
cuadrados, la vista— **no está en nuestras 7 columnas**.

Así se reporta un resultado nulo, sin adornos y sin disculpas:

> *"Ninguna de las variables disponibles explica el precio de forma apreciable. Las correlaciones con
> número de camas (0.15), calificación (0.20) y cantidad de reseñas (−0.03) son todas débiles o nulas.
> Los determinantes probables del precio —ubicación, temporada, nivel de lujo, superficie— no están
> presentes en este dataset."*

> 💡 **La honestidad intelectual también es parte del método.** Reportar "no encontramos relación" es
> mucho más valioso que forzar una conclusión que los datos no sostienen.

### 💼 Y esto tiene una consecuencia práctica inmediata

Guardá este resultado, porque **te acabás de ahorrar trabajo futuro**.

En la **Clase 3 de este módulo (08/08)** vamos a construir una **regresión**: un modelo que predice una
variable numérica a partir de otras. La correlación es **el paso previo obligatorio** de ese proceso:
sin correlación, no hay modelo lineal útil.

Ya sabemos que un modelo para predecir el precio de un Airbnb usando solo estas variables **va a
funcionar mal**. Y lo supimos **antes de construirlo**, con `.corr()` y quince segundos de trabajo.

> **Para eso sirve el análisis exploratorio: para ahorrarte el trabajo de modelar lo que no tiene señal.**

## ⚠️ Y la advertencia clásica que no puede faltar

> ## Correlación ≠ Causalidad

Que dos variables se muevan juntas **no significa que una cause la otra**.

El ejemplo canónico: el consumo de helado y los ahogamientos en piscinas están fuertemente correlacionados.
El helado no ahoga a nadie. Hay una tercera variable —**el calor**— que causa ambos. A esa tercera
variable se la llama **variable de confusión**.

Si mañana encontrás que el precio correlaciona con el rating, **no concluyas que subir el precio mejora
la calificación**. Podría ser que las propiedades caras sean genuinamente mejores, o que quien paga más
califique más generosamente, o cualquier otra cosa.

## ✍️ TU TURNO 9

Hacé un **scatter plot de `camas` (eje X) contra `Price(in dollar)` (eje Y)**.

🤔 Sabiendo que la correlación es 0.149, ¿esperás ver una nube ordenada en diagonal o un manchón sin forma?

💡 *Pista: usá `plt.scatter()` con `alpha=0.5`. No olvides las etiquetas de los ejes.*


In [ ]:
# TU TURNO 9: scatter de camas vs precio
# Tu código aquí



---

## 🗺️ Tabla de decisión: ¿qué gráfico uso?

La pregunta que todo el mundo hace al final. Esta tabla es la respuesta corta, y te va a servir mucho
más allá de esta clase.

| Tengo... | Quiero saber... | Uso... |
|---|---|---|
| 1 variable numérica | Su forma y distribución | **Histograma** |
| 1 variable numérica | Centro, dispersión y atípicos, compacto | **Boxplot** |
| 1 variable categórica | Cuántos hay de cada una | **Gráfico de barras** |
| 1 variable categórica (2-3 valores) | Qué proporción del total | **Torta** (con reservas) |
| 1 numérica + 1 categórica | Comparar grupos | **Boxplot comparativo** |
| 2 variables numéricas | Si se relacionan | **Scatter plot** |
| 1 numérica en el tiempo | Evolución | Gráfico de líneas |
| Muchas variables numéricas | Todas las correlaciones | Mapa de calor de correlación |

### Los 7 gráficos que hicimos hoy y qué reveló cada uno

| Gráfico | Lo que reveló |
|---|---|
| Histograma | **Por qué** la media y la mediana difieren |
| Boxplot | Que **todos** los outliers están del lado caro |
| Histograma + curva normal | Que el modelo teórico predice **precios negativos** |
| Barras | Que Airbnb es un negocio de **alojamientos chicos** |
| Torta | Que hay gráficos que **piden más de lo que dan** |
| Boxplot comparativo | Que **cuanto más caro el segmento, más impredecible** el precio |
| Scatter | Que **443 datos habían desaparecido** sin que nadie avisara |

**Ninguno de esos siete hallazgos aparece en una tabla de estadísticos.**

### 📖 El decálogo de la visualización honesta

1. **El eje Y de un gráfico de barras empieza en cero.** Sin excepciones.
2. **Siempre etiquetá los ejes, con unidades.** Un número sin unidad no es información.
3. **Preguntá cuántos datos tiene el gráfico.** Los `NaN` desaparecen en silencio.
4. **Ordená las categorías por valor**, salvo que tengan orden natural.
5. **Nunca uses 3D.** Ni en barras, ni en tortas. La perspectiva distorsiona.
6. **Un outlier no es un error.** Antes de borrarlo, entendelo. Si lo borrás, decilo.
7. **Elegí el gráfico según la pregunta**, no según cuál te salió lindo.
8. **Correlación no es causalidad.** Ni aunque la nube esté perfectamente alineada.
9. **Informá el tamaño de muestra de cada grupo.** Una caja de 3 datos se ve igual que una de 300.
10. **Graficá siempre antes de resumir.**

> 💡 **¿Por qué la 10?** En 1973 el estadístico **Francis Anscombe** construyó cuatro conjuntos de datos
> con la **misma media, la misma varianza, la misma correlación y la misma recta de regresión** — y al
> graficarlos resultaron ser cuatro realidades completamente distintas. Se lo conoce como el
> **Cuarteto de Anscombe**, y es la justificación canónica de por qué siempre hay que graficar.
>
> **Los números resumen. Los gráficos revelan.**


---

# 10. Cierre: lo que aprendimos hoy

## ✅ La respuesta a la pregunta ancla

> ### ¿Cuánto cuesta realmente un Airbnb?

**La respuesta profesional completa:**

> *"En este dataset de 538 anuncios, la mediana del precio es **$138 por noche**, con un rango
> intercuartílico de **$90 a $222**. La media ($175) es un 27% superior a la mediana debido a una
> asimetría positiva fuerte (2.34) causada por 28 anuncios de lujo. **Reportamos la mediana** por ser
> robusta ante esos valores extremos. Existe variación importante según el tipo de propiedad: desde
> $100 (habitación) hasta $222 (villa)."*

Comparalo con la respuesta de alguien que solo sabe apretar `.mean()`:

> *"Cuesta $175."*

**Esa es la diferencia entre ejecutar funciones y hacer estadística.**

## 📋 Lo que dominás ahora

| Concepto | Función | Cuándo usarlo |
|---|---|---|
| Tabla de frecuencias | `value_counts()` | Variables categóricas |
| Media | `.mean()` | Datos simétricos, sin extremos |
| **Mediana** | `.median()` | **Datos con outliers o cola** |
| Moda | `.mode()` | Categóricas, o el valor más común |
| Rango | `.max() - .min()` | Vistazo rápido, poco confiable |
| Varianza | `.var()` | Paso intermedio (unidades al cuadrado) |
| Desviación estándar | `.std()` | **La medida de dispersión estándar** |
| Cuartiles | `.quantile()` | Describir el rango típico |
| Resumen completo | `.describe()` | Punto de partida siempre |
| Asimetría | `.skew()` | Decidir entre media y mediana |
| Comparar grupos | `.groupby()` | La función más poderosa de pandas |
| Correlación | `.corr()` | Relación entre numéricas |

## 🧠 Las 8 ideas que valen más que las funciones

1. **Nunca calcules sobre datos que no miraste.** `head()` e `info()` primero. Siempre.
2. **La media es sensible a extremos; la mediana es robusta.** Un solo penthouse movió el promedio $92.
3. **Un promedio sin dispersión al lado es información incompleta.**
4. **Una variable que no varía, no informa.** El rating de Airbnb es casi inútil: todos tienen 4.9.
5. **Siempre mirá `count` antes de creerle a una media.** Tres observaciones no son un resultado.
6. **Un gráfico nunca avisa cuántos datos le faltan.** 443 puntos desaparecieron sin decir nada.
7. **Correlación no es causalidad.** Nunca. Ni aunque el gráfico sea muy convincente.
8. **Pandas ejecuta lo que le pidas, aunque no tenga sentido.** El único filtro contra un dato absurdo
   sos vos, con tu conocimiento del negocio.

## ⚠️ La advertencia final: describir ≠ generalizar

Todo lo que calculamos hoy describe a **estos 538 anuncios**. Nada más.

No sabemos cómo fueron elegidos. Podrían no representar a Airbnb en general, ni a ningún país en
particular. **Estadística descriptiva describe la muestra que tenés en la mano.**

Extender conclusiones desde una muestra hacia una población entera —y cuantificar cuánta confianza
merece esa extensión— es exactamente el tema del próximo módulo: **estadística inferencial**.

Y ahí vamos a necesitar todo lo de hoy, especialmente la parte donde descubrimos que **el precio no
sigue una distribución normal**.

---

# 📝 Tarea

Completá el **`Descriptive_Statistics_assignment1.ipynb`** del repositorio oficial de Khipus
(carpeta `1 Descriptive_Statistics`).

Trabaja con **`sales_data.csv`**, un dataset distinto, y te pide exactamente lo que practicamos hoy:
media, mediana, desviación estándar, rango, percentiles, histograma y boxplot.

**Buenas noticias:** el assignment trae los **resultados esperados** escritos. Podés verificarte solo.

💡 **Antes de empezar, aplicá el método de hoy:** cargá el archivo, hacé `head()` e `info()`, revisá
nulos y duplicados. **No calcules nada hasta haber mirado los datos.**

## 📚 Material de repaso

En el repositorio también está **`Descriptive_Statistics_Case_Study.ipynb`**, el notebook oficial sobre
este mismo dataset de Airbnb. Es más corto y directo que el nuestro — sirve como referencia limpia para
consultar las funciones.

Vas a notar que **no hace la limpieza que hicimos hoy**. Ahora ya sabés qué se está perdiendo.

---

<div style="text-align:center; padding:20px;">

### 🎓 Fin de la clase

**Módulo 2 — Estadística Aplicada con Python**
Docente: Walter J. Méndez · UTEPSA / Khipus.ai

*"El valor no está en tener datos, sino en saber convertirlos en decisiones inteligentes."*

</div>